In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os

In [2]:
load_dotenv("../.env")
census_key = os.getenv("CENSUS_API_KEY")

In [3]:
API_KEY    = census_key
STATE_FIPS = "54"                          # West Virginia
YEARS      = [2018, 2019, 2020, 2021, 2022, 2023]

# S1201_C02_001E = Percent never married
# S1201_C05_001E = Percent divorced
VARIABLES  = "S1201_C02_001E,S1201_C03_001E"

In [4]:
# ── STEP 1: Pull ACS 5-Year S1201 for all WV counties ───────────────────────
records = []

for year in YEARS:
    url = (
        f"https://api.census.gov/data/{year}/acs/acs5/subject"
        f"?get=NAME,{VARIABLES}"
        f"&for=county:*"
        f"&in=state:{STATE_FIPS}"
        f"&key={API_KEY}"
    )
    response = requests.get(url)

    # Debug print in case of errors
    print(f"Year: {year} | Status: {response.status_code}")
    if response.status_code != 200:
        print(f"Response: {response.text[:300]}")
        continue

    data = response.json()
    headers = data[0]
    for row in data[1:]:
        record = dict(zip(headers, row))
        record["Year"] = year
        records.append(record)

df = pd.DataFrame(records)
df

Year: 2018 | Status: 200
Year: 2019 | Status: 200
Year: 2020 | Status: 200
Year: 2021 | Status: 200
Year: 2022 | Status: 200
Year: 2023 | Status: 200


,NAME,S1201_C02_001E,S1201_C03_001E,state,county,Year
0,"Grant County, West Virginia",51.6,8.5,54,023,2018
1,"Hampshire County, West Virginia",48.1,7.5,54,027,2018
2,"Brooke County, West Virginia",50.3,8.4,54,009,2018
3,"Doddridge County, West Virginia",48.3,7.5,54,017,2018
4,"Hardy County, West Virginia",62.0,5.8,54,031,2018
...,...,...,...,...,...,...
325,"Webster County, West Virginia",54.3,8.6,54,101,2023
326,"Wetzel County, West Virginia",49.1,9.6,54,103,2023
327,"Wirt County, West Virginia",57.8,10.4,54,105,2023
328,"Wood County, West Virginia",48.6,7.4,54,107,2023


In [5]:
df["FIPS_Code"]        = df["state"] + df["county"]
df["County"]           = df["NAME"].str.replace(", West Virginia", "", regex=False)
df["Pct_Never_Married"] = pd.to_numeric(df["S1201_C02_001E"], errors="coerce")
df["Pct_Divorced"]      = pd.to_numeric(df["S1201_C03_001E"], errors="coerce")

# ── STEP 3: Final structure ───────────────────────────────────────────────────
df = df[["Year", "FIPS_Code", "County", "Pct_Never_Married", "Pct_Divorced"]].copy()
df = df.sort_values(["Year", "FIPS_Code"]).reset_index(drop=True)

df

,Year,FIPS_Code,County,Pct_Never_Married,Pct_Divorced
0,2018,54001,Barbour County,50.3,7.9
1,2018,54003,Berkeley County,52.4,5.5
2,2018,54005,Boone County,53.1,9.1
3,2018,54007,Braxton County,56.0,7.1
4,2018,54009,Brooke County,50.3,8.4
...,...,...,...,...,...
325,2023,54101,Webster County,54.3,8.6
326,2023,54103,Wetzel County,49.1,9.6
327,2023,54105,Wirt County,57.8,10.4
328,2023,54107,Wood County,48.6,7.4


In [6]:
df.to_csv("marital_status.csv", index=False)